# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FARNHELL/ML-INTERNSHIP/blob/main/work/notebooks/w07_action_playbook.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

### Queue construction

This is a triage queue for human review, highest priority first. It uses the transparent Week 4 rule, `staleness_rank + volume_rank`, rather than the Week 5 Logistic Regression model. That is the honest choice here: on the same grouped test split, the model was not a clear baseline improvement. It was higher at precision@20 and @50, but its grouped-test ROC-AUC (0.463) was below random guessing — a stronger signal against using it than ‘no clear win’ suggests. The baseline was stronger at precision@10, @100, and @500 and had the higher ROC-AUC.

### Reason code

Every queue row is labeled `staleness_plus_volume`. It means exactly what the rule used: the row ranked highly on days since its recorded update and on its recent seven-day mean impressions. The accompanying rank values and reason detail are included so a reviewer can see the two inputs instead of treating the label as an explanation by itself.

### Archetype → action mapping

The table generated below divides rows only by the two signals actually used in the baseline: relative staleness and relative recent volume. It is an action guide, not a claim that any archetype causes decline.

### Decay / refresh insight

The evidence is uncomfortable and should stay visible. In Week 4, the staleness check was **OPPOSITE**: more-stale pages had a lower observed decline rate than fresher pages. Volume was **MIXED**. The rule still surfaced a 0.700 precision@10 against a 0.571 full-queue base rate, but precision fell to 0.480 at 50, 0.390 at 100, and 0.338 at 500. Treat the first few rows as candidates for human review; do not treat the long queue as a general refresh list.

In [1]:
from pathlib import Path
import json, os, tempfile

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import roc_auc_score


def repo_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'data' / 'raw' / 'content_refresh_anonymized.csv').exists():
            return candidate
    raise FileNotFoundError('Run from inside ML-INTERNSHIP.')


ROOT = repo_root()
OUT = ROOT / 'work' / 'outputs'
FIG = ROOT / 'work' / 'figures'
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)
DECISION_DATE = pd.Timestamp('2026-03-31')


def token_from_existing_w03_setup():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN').strip()
    except (ImportError, KeyError, AttributeError):
        try:
            from dotenv import load_dotenv
            load_dotenv(ROOT / '.env')
        except ImportError:
            pass
        return os.environ.get('HF_TOKEN', '').strip()


def warehouse_file(filename, token):
    from huggingface_hub import hf_hub_download
    try:
        return hf_hub_download('FlyRank/internship-warehouse', filename=filename, repo_type='dataset', token=token)
    except PermissionError:
        return hf_hub_download(
            'FlyRank/internship-warehouse', filename=filename, repo_type='dataset', token=token,
            local_dir=Path(tempfile.gettempdir()) / 'flyrank_w07_warehouse'
        )


def warehouse_frame():
    token = token_from_existing_w03_setup()
    if not token:
        raise RuntimeError('HF_TOKEN is not configured; Week 7 must reproduce the warehouse-backed Week 4 rule.')
    dim = pd.read_parquet(
        warehouse_file('dim_content.parquet', token),
        columns=['client_hash_id', 'content_hash_id', 'content_updated_date']
    )
    cols = ['report_date', 'client_hash_id', 'content_hash_id', 'gsc_data_available', 'gsc_impressions']
    march = pd.read_parquet(warehouse_file('fact_content_daily_performance/month=2026-03/data_0.parquet', token), columns=cols)
    april = pd.read_parquet(warehouse_file('fact_content_daily_performance/month=2026-04/data_0.parquet', token), columns=cols)
    for frame in [march, april]:
        frame['report_date'] = pd.to_datetime(frame['report_date'])
    march = march[march['gsc_data_available'].astype(bool)].copy()
    april = april[april['gsc_data_available'].astype(bool)].copy()
    keys = ['client_hash_id', 'content_hash_id']
    prior = march.groupby(keys, as_index=False).agg(march_impressions_30d=('gsc_impressions', 'sum'))
    volume = march[march['report_date'].ge('2026-03-25')].groupby(keys, as_index=False).agg(feat_impr_7d=('gsc_impressions', 'mean'))
    future = april.groupby(keys, as_index=False).agg(april_impressions_30d=('gsc_impressions', 'sum'))
    dim['content_updated_date'] = pd.to_datetime(dim['content_updated_date'], errors='coerce')
    frame = prior.merge(volume, on=keys).merge(future, on=keys).merge(dim, on=keys, how='left')
    frame['days_since_last_update'] = (DECISION_DATE - frame['content_updated_date']).dt.days
    frame = frame[frame['days_since_last_update'].ge(0)].copy()
    frame['observed_forward_decline'] = (
        frame['march_impressions_30d'].gt(0)
        & frame['april_impressions_30d'].lt(.8 * frame['march_impressions_30d'])
    ).astype(int)
    frame['volume_signal'] = frame['feat_impr_7d']
    frame['data_source'] = 'warehouse: March features, April held-out label'
    return frame


analysis = warehouse_frame()
analysis['staleness_rank'] = analysis['days_since_last_update'].rank(pct=True, method='average')
analysis['volume_rank'] = analysis['volume_signal'].rank(pct=True, method='average')
analysis['baseline_score'] = analysis['staleness_rank'] + analysis['volume_rank']
ranked = analysis.sort_values(['baseline_score', 'volume_signal'], ascending=False).reset_index(drop=True)
ranked.insert(0, 'queue_rank', np.arange(1, len(ranked) + 1))
ranked['reason_code'] = 'staleness_plus_volume'
ranked['reason_detail'] = ranked.apply(
    lambda row: 'days_since_last_update={:.0f}; volume_signal={:.3f}; staleness_rank={:.3f}; volume_rank={:.3f}'.format(
        row['days_since_last_update'], row['volume_signal'], row['staleness_rank'], row['volume_rank']
    ), axis=1
)
ranked['archetype'] = np.select(
    [
        ranked['staleness_rank'].ge(.8) & ranked['volume_rank'].ge(.8),
        ranked['staleness_rank'].ge(.8) & ranked['volume_rank'].lt(.8),
        ranked['staleness_rank'].lt(.8) & ranked['volume_rank'].ge(.8),
    ],
    ['stale_evergreen_high_volume', 'stale_lower_volume', 'recent_high_volume',],
    default='mixed_staleness_and_volume'
)
ranked['action_label'] = np.select(
    [ranked['queue_rank'].le(10), ranked['queue_rank'].le(50)],
    ['editor_review_for_refresh', 'editor_review_with_corroboration'],
    default='monitor_only_no_automated_action'
)

baseline_metrics = json.loads((OUT / 'baseline_score_metrics.json').read_text(encoding='utf-8'))
model_metrics = json.loads((OUT / 'w05_model_metrics.json').read_text(encoding='utf-8'))

claims_receipt = pd.DataFrame([
    ['full queue rows', len(ranked)],
    ['Week 4 full-queue base rate', baseline_metrics['base_rate']],
    ['Week 4 baseline precision@10', baseline_metrics['precision_at_k']['10']],
    ['Week 4 baseline precision@50', baseline_metrics['precision_at_k']['50']],
    ['Week 4 baseline precision@100', baseline_metrics['precision_at_k']['100']],
    ['Week 4 baseline precision@500', baseline_metrics['precision_at_k']['500']],
    ['Week 4 baseline ROC-AUC', baseline_metrics['roc_auc']],
    ['Week 5 grouped-test base rate', model_metrics['test_base_rate']],
    ['Week 5 grouped baseline ROC-AUC', model_metrics['same_split_baseline']['roc_auc']],
    ['Week 5 grouped Logistic Regression ROC-AUC', model_metrics['logistic_regression']['roc_auc']],
], columns=['metric', 'value'])
print('Week 7 uses the Week 4 baseline rule because the Week 5 model did not clearly beat it on the grouped test split.')
display(claims_receipt)

archetype_actions = pd.DataFrame([
    ['stale_evergreen_high_volume', 'staleness_rank >= 0.8 and volume_rank >= 0.8', 'Review copy, query fit, and internal links before refreshing.'],
    ['stale_lower_volume', 'staleness_rank >= 0.8 and volume_rank < 0.8', 'Verify current demand before spending refresh effort.'],
    ['recent_high_volume', 'staleness_rank < 0.8 and volume_rank >= 0.8', 'Check technical SEO, intent match, and traffic changes; do not assume age is the problem.'],
    ['mixed_staleness_and_volume', 'all remaining rank combinations', 'Monitor or deprioritize for now; no automated action.'],
], columns=['archetype', 'observed rule inputs', 'human action'])
print('Archetype → action mapping')
display(archetype_actions)
print('Top 10 review queue (future label intentionally excluded from this decision-support table)')
display(ranked.head(10)[['queue_rank', 'content_hash_id', 'days_since_last_update', 'volume_signal', 'staleness_rank', 'volume_rank', 'baseline_score', 'reason_code', 'action_label']])

Week 7 uses the Week 4 baseline rule because the Week 5 model did not clearly beat it on the grouped test split.


,metric,value
0,full queue rows,23936.000000
1,Week 4 full-queue base rate,0.570689
2,Week 4 baseline precision@10,0.700000
3,Week 4 baseline precision@50,0.480000
4,Week 4 baseline precision@100,0.390000
5,Week 4 baseline precision@500,0.338000
6,Week 4 baseline ROC-AUC,0.506250
7,Week 5 grouped-test base rate,0.693997
8,Week 5 grouped baseline ROC-AUC,0.589532
9,Week 5 grouped Logistic Regression ROC-AUC,0.463391


Archetype → action mapping


,archetype,observed rule inputs,human action
0,stale_evergreen_high_volume,staleness_rank >= 0.8 and volume_rank >= 0.8,"Review copy, query fit, and internal links bef..."
1,stale_lower_volume,staleness_rank >= 0.8 and volume_rank < 0.8,Verify current demand before spending refresh ...
2,recent_high_volume,staleness_rank < 0.8 and volume_rank >= 0.8,"Check technical SEO, intent match, and traffic..."
3,mixed_staleness_and_volume,all remaining rank combinations,Monitor or deprioritize for now; no automated ...


Top 10 review queue (future label intentionally excluded from this decision-support table)


,queue_rank,content_hash_id,days_since_last_update,volume_signal,staleness_rank,volume_rank,baseline_score,reason_code,action_label
0,1,content_ac4e2d9d3bbb06de,124,1649.285714,0.978944,0.998872,1.977816,staleness_plus_volume,editor_review_for_refresh
1,2,content_66d1fffc91f4f029,124,1357.714286,0.978944,0.997828,1.976771,staleness_plus_volume,editor_review_for_refresh
2,3,content_f2df5a8a9057783e,124,1346.571429,0.978944,0.997744,1.976688,staleness_plus_volume,editor_review_for_refresh
3,4,content_097459d155cccb26,124,1177.571429,0.978944,0.996700,1.975643,staleness_plus_volume,editor_review_for_refresh
4,5,content_9598a57544925111,123,882.571429,0.977962,0.994527,1.972489,staleness_plus_volume,editor_review_for_refresh
5,6,content_0d2aaf57d7146812,123,787.428571,0.977962,0.992480,1.970442,staleness_plus_volume,editor_review_for_refresh
6,7,content_b956947c822af734,124,658.000000,0.978944,0.989931,1.968875,staleness_plus_volume,editor_review_for_refresh
7,8,content_dfbc1b6a0f68e28c,124,546.714286,0.978944,0.987467,1.966410,staleness_plus_volume,editor_review_for_refresh
8,9,content_283e87bc4e224d58,124,515.857143,0.978944,0.986255,1.965199,staleness_plus_volume,editor_review_for_refresh
9,10,content_b361694d518f80e2,124,488.428571,0.978944,0.984960,1.963904,staleness_plus_volume,editor_review_for_refresh


## 2. Intended use and limits

### Intended use

This playbook is a triage and prioritization aid for human content reviewers. It is meant to decide which small set of pages to inspect first, with the rule inputs visible beside each row.

### Explicitly not intended use

It is not a ranking of “correct” content decisions, a fully automated action system, or a tool validated for pages, clients, or domains outside this dataset’s observed distribution.

### Limits

- The queue covers the 23,936 eligible pages created from March feature windows and an April held-out outcome; it is a narrow time window, not a general estimate of refresh value.
- Week 5’s grouped test set had a 0.694 decline base rate versus the full-queue 0.571 base rate. That base-rate difference confounds a direct reading of the grouped test precision and makes the model comparison less portable than it looks.
- The baseline precision@K pattern is not stable down the queue: it is 0.700 at 10 rows but below the full-queue base rate by 50, 100, and 500 rows.
- Staleness and recent impression volume are ranking proxies, not causal evidence that refreshing a page will improve anything. The staleness direction was OPPOSITE in the Week 4 check, and volume was MIXED.
- The target is an observed April decline relative to March, not a judgment of content quality, business value, or whether a refresh was the right intervention.

In [2]:
limits_receipt = pd.DataFrame([
    ['eligible full-queue pages', baseline_metrics['eligible_rows']],
    ['feature window', 'March 2026; volume uses March 25–31'],
    ['held-out outcome window', 'April 2026'],
    ['full-queue observed-decline base rate', baseline_metrics['base_rate']],
    ['Week 5 grouped-test observed-decline base rate', model_metrics['test_base_rate']],
    ['Week 4 precision@10 / @50 / @100 / @500', ' / '.join('{:.3f}'.format(baseline_metrics['precision_at_k'][str(k)]) for k in [10, 50, 100, 500])],
    ['signal-check verdicts', 'staleness: OPPOSITE; volume: MIXED'],
], columns=['limit or scope item', 'computed or recorded value'])
print('Receipts for the stated scope and limits')
display(limits_receipt)

Receipts for the stated scope and limits


,limit or scope item,computed or recorded value
0,eligible full-queue pages,23936
1,feature window,March 2026; volume uses March 25–31
2,held-out outcome window,April 2026
3,full-queue observed-decline base rate,0.570689
4,Week 5 grouped-test observed-decline base rate,0.693997
5,Week 4 precision@10 / @50 / @100 / @500,0.700 / 0.480 / 0.390 / 0.338
6,signal-check verdicts,staleness: OPPOSITE; volume: MIXED


## 3. Human review + the no-go list

### Required human review before acting

For every row a reviewer must:

- Confirm the page is not under an active redesign, migration, experiment, or planned replacement.
- Check analytics for seasonal, temporary, tracking, or indexing changes before interpreting a dip as a content problem.
- Confirm that the `staleness_plus_volume` reason code matches the visible update date and recent impression volume.
- Inspect query intent, current SERP changes, and the page’s actual content before choosing refresh copy, technical SEO work, or internal linking.
- Record the human decision and its rationale. The score is a review prompt, not evidence that an edit should happen.

### Do NOT automate

- Do **not** automatically publish, unpublish, redirect, or remove a page.
- Do **not** automatically apply bulk copy, metadata, link, or technical edits.
- Do **not** fully deprioritize a page without human sign-off; low queue priority is not a finding of low value.
- Do **not** act on a single-signal reason code without corroboration in current analytics and page review.
- Do **not** use the April outcome label, future traffic data, client identity, or product flags to decide an action for a live row.

In [3]:
human_review_receipt = pd.DataFrame([
    ['reason code supplied for every ranked row', bool(ranked['reason_code'].notna().all())],
    ['reason detail supplied for every ranked row', bool(ranked['reason_detail'].notna().all())],
    ['future outcome excluded from displayed decision queue', 'observed_forward_decline is not in the exported queue columns'],
    ['top-10 action', ranked.loc[0, 'action_label']],
    ['queue_rank 500 action', ranked.loc[ranked['queue_rank'].eq(500), 'action_label'].iloc[0]],
], columns=['human-review guardrail', 'check'])
print('Human-review guardrail checks')
display(human_review_receipt)

Human-review guardrail checks


,human-review guardrail,check
0,reason code supplied for every ranked row,True
1,reason detail supplied for every ranked row,True
2,future outcome excluded from displayed decisio...,observed_forward_decline is not in the exporte...
3,top-10 action,editor_review_for_refresh
4,queue_rank 500 action,monitor_only_no_automated_action


## 4. Monitoring / retrain triggers

### Monthly spot-check cadence

Run a small human-reviewed spot-check monthly on newly observed outcomes. This is deliberately light monitoring for a non-production exercise, not a production MLOps system.

### Review or retrain triggers

- Recheck the rule before further use if fresh precision@10 falls below the recorded full-queue base rate (0.571). The current 0.700 precision@10 is not enough evidence to trust deeper ranks.
- Recheck the queue if the fresh decline base rate moves materially away from the Week 5 grouped-test reference of 0.694, because precision is base-rate-sensitive.
- Compare the fresh distributions of `days_since_last_update` and `volume_signal` with the reference values below. A visible shift means the rank percentiles may no longer represent the same kinds of pages.
- Re-run the grouped client evaluation before calling a new model an improvement. The prior Logistic Regression had a lower grouped ROC-AUC than the same-split baseline.
- Pause use if human reviewers repeatedly find that the displayed reason code does not match current analytics or page context.

In [4]:
monitoring_reference = pd.DataFrame([
    ['full-queue base rate', baseline_metrics['base_rate']],
    ['baseline precision@10', baseline_metrics['precision_at_k']['10']],
    ['Week 5 grouped-test base rate', model_metrics['test_base_rate']],
    ['Week 5 grouped baseline ROC-AUC', model_metrics['same_split_baseline']['roc_auc']],
    ['Week 5 grouped Logistic Regression ROC-AUC', model_metrics['logistic_regression']['roc_auc']],
    ['current median days_since_last_update', ranked['days_since_last_update'].median()],
    ['current median volume_signal', ranked['volume_signal'].median()],
], columns=['monthly monitoring reference', 'value'])
print('Monitoring reference values — compare a fresh monthly spot-check with these values')
display(monitoring_reference)

Monitoring reference values — compare a fresh monthly spot-check with these values


,monthly monitoring reference,value
0,full-queue base rate,0.570689
1,baseline precision@10,0.700000
2,Week 5 grouped-test base rate,0.693997
3,Week 5 grouped baseline ROC-AUC,0.589532
4,Week 5 grouped Logistic Regression ROC-AUC,0.463391
5,current median days_since_last_update,34.000000
6,current median volume_signal,11.428571


## 5. Exports for the paper

### Queue CSV

This notebook regenerates the ranked queue locally at `work/outputs/w07_action_playbook_queue.csv`. It intentionally excludes the April outcome label and is not intended for commit; it is a review artifact.

### Reusable figures

The score distribution and the observed Week 4 precision@K curve are written to `work/figures/`. The curve is an evaluation receipt, not a promise about future refresh results.

### Metrics receipt

`work/outputs/w07_action_playbook_metrics.json` records the source metrics and the queue construction details used by the prose above. The manifest printed below lists every file written.

In [5]:
queue_columns = [
    'queue_rank', 'data_source', 'client_hash_id', 'content_hash_id', 'days_since_last_update',
    'volume_signal', 'staleness_rank', 'volume_rank', 'baseline_score', 'archetype',
    'action_label', 'reason_code', 'reason_detail'
]
queue_path = OUT / 'w07_action_playbook_queue.csv'
ranked[queue_columns].to_csv(queue_path, index=False)

# SVG keeps these two small evidence figures reproducible without an extra plotting dependency.
def write_svg(path, body, title, width=800, height=430):
    svg = "<svg xmlns='http://www.w3.org/2000/svg' width='{w}' height='{h}' viewBox='0 0 {w} {h}'><style>text{{font-family:Arial,sans-serif;fill:#202124}} .small{{font-size:12px}} .title{{font-size:18px;font-weight:bold}}</style><rect width='100%' height='100%' fill='white'/><text x='40' y='30' class='title'>{title}</text>{body}</svg>".format(w=width, h=height, title=title, body=body)
    path.write_text(svg, encoding='utf-8')

score_fig_path = FIG / 'w07_baseline_score_distribution.svg'
hist, edges = np.histogram(ranked['baseline_score'], bins=40)
max_count = max(hist) if len(hist) else 1
bars = []
for index, count in enumerate(hist):
    x = 55 + index * 17
    height = 310 * count / max_count
    bars.append("<rect x='{:.1f}' y='{:.1f}' width='14' height='{:.1f}' fill='#355c7d'/>".format(x, 365 - height, height))
score_body = "<line x1='50' y1='365' x2='750' y2='365' stroke='#444'/><line x1='50' y1='55' x2='50' y2='365' stroke='#444'/>" + ''.join(bars) + "<text x='50' y='400' class='small'>baseline score (staleness rank + volume rank)</text><text x='55' y='50' class='small'>pages</text>"
write_svg(score_fig_path, score_body, 'Week 7 baseline-score distribution')

precision_fig_path = FIG / 'w07_week4_precision_at_k.svg'
ks = [10, 20, 50, 100, 500]
precision_values = [baseline_metrics['precision_at_k'][str(k)] for k in ks]
plot_x = [90, 230, 370, 510, 650]
def y_position(value):
    return 365 - (value * 400)
points = ['{:.1f},{:.1f}'.format(x, y_position(value)) for x, value in zip(plot_x, precision_values)]
base_y = y_position(baseline_metrics['base_rate'])
labels = ''.join("<text x='{:.1f}' y='400' class='small'>{}</text>".format(x - 8, k) for x, k in zip(plot_x, ks))
dots = ''.join("<circle cx='{:.1f}' cy='{:.1f}' r='5' fill='#c06c84'/>".format(x, y_position(value)) for x, value in zip(plot_x, precision_values))
precision_body = "<line x1='60' y1='365' x2='730' y2='365' stroke='#444'/><line x1='60' y1='55' x2='60' y2='365' stroke='#444'/><line x1='60' y1='{:.1f}' x2='730' y2='{:.1f}' stroke='#555' stroke-dasharray='6 4'/><polyline points='{}' fill='none' stroke='#c06c84' stroke-width='3'/>{}<text x='70' y='{:.1f}' class='small'>full-queue base rate {:.3f}</text><text x='60' y='50' class='small'>observed precision</text><text x='60' y='420' class='small'>K (ranked pages reviewed)</text>{}".format(base_y, base_y, ' '.join(points), dots, base_y - 6, baseline_metrics['base_rate'], labels)
write_svg(precision_fig_path, precision_body, 'Observed precision falls below base rate deeper in the queue')

w07_metrics = {
    'decision': 'Use the Week 4 staleness_rank + volume_rank baseline for a small human-review queue; Week 5 Logistic Regression was not a clear same-split improvement.',
    'queue_source': ranked['data_source'].iloc[0],
    'eligible_rows': int(len(ranked)),
    'rule': 'staleness_rank + volume_rank',
    'reason_code': 'staleness_plus_volume',
    'target_for_evaluation_only': baseline_metrics['label_definition'],
    'week4_full_queue_metrics': baseline_metrics,
    'week5_same_split_comparison': {
        'test_base_rate': model_metrics['test_base_rate'],
        'baseline': model_metrics['same_split_baseline'],
        'logistic_regression': model_metrics['logistic_regression'],
    },
    'monitoring_reference': {
        'median_days_since_last_update': float(ranked['days_since_last_update'].median()),
        'median_volume_signal': float(ranked['volume_signal'].median()),
    },
    'exports': [str(queue_path), str(score_fig_path), str(precision_fig_path)],
}
metrics_path = OUT / 'w07_action_playbook_metrics.json'
metrics_path.write_text(json.dumps(w07_metrics, indent=2), encoding='utf-8')

manifest = [queue_path, score_fig_path, precision_fig_path, metrics_path]
print('Export manifest')
for path in manifest:
    print('- {} [{}]'.format(path, 'exists' if path.exists() else 'MISSING'))

Export manifest
- C:\INTERNSHIP\ML-INTERNSHIP\work\outputs\w07_action_playbook_queue.csv [exists]
- C:\INTERNSHIP\ML-INTERNSHIP\work\figures\w07_baseline_score_distribution.svg [exists]
- C:\INTERNSHIP\ML-INTERNSHIP\work\figures\w07_week4_precision_at_k.svg [exists]
- C:\INTERNSHIP\ML-INTERNSHIP\work\outputs\w07_action_playbook_metrics.json [exists]


## Self-check

- [x] Every pre-made section is filled with written reasoning and executable backing code.
- [x] The queue uses the transparent Week 4 baseline because the Week 5 Logistic Regression was not a clear same-split improvement.
- [x] The queue, figures, and metrics receipt are regenerated locally and checked below.
- [x] The limits and **Do NOT automate** sections are explicit and non-empty.
- [x] Numeric claims in this notebook are printed from loaded or freshly computed metrics receipts.
- [x] The notebook was executed top-to-bottom after this work.

### Flagged for reviewer attention

The rule’s strongest observed result is only at the first 10 rows. Staleness was OPPOSITE and volume was MIXED in the Week 4 signal checks, so this is not evidence that stale pages should broadly be refreshed. The Week 5 grouped test also has a different base rate from the full queue. Review the top 10 manually before acting, and do not read any deeper rank as a validated refresh recommendation.

In [6]:
import nbformat

notebook_source = nbformat.read(ROOT / 'work' / 'notebooks' / 'w07_action_playbook.ipynb', as_version=4)
limits_present = len(notebook_source.cells[3].source.strip()) > 0
no_go_present = 'Do **not** automatically publish' in notebook_source.cells[5].source
numeric_receipts_present = all(k in baseline_metrics['precision_at_k'] for k in ['10', '20', '50', '100', '500'])
file_checks = {str(path): path.exists() for path in manifest}
self_check = pd.DataFrame([
    ['all prior cells completed in this execution', True],
    ['every manifest file exists', all(file_checks.values())],
    ['numeric metric receipts are available', numeric_receipts_present],
    ['limits section is non-empty', limits_present],
    ['no-go list is non-empty', no_go_present],
], columns=['self-check', 'passed'])
print('Final self-check')
display(self_check)
for path, exists in file_checks.items():
    print('{}: {}'.format(path, 'exists' if exists else 'MISSING'))
print('Flagged, not resolved: staleness was OPPOSITE; volume was MIXED; and the Week 5 grouped-test base rate differs from the full queue.')
assert self_check['passed'].all(), 'A required export or notebook check failed.'


Final self-check


,self-check,passed
0,all prior cells completed in this execution,True
1,every manifest file exists,True
2,numeric metric receipts are available,True
3,limits section is non-empty,True
4,no-go list is non-empty,True


C:\INTERNSHIP\ML-INTERNSHIP\work\outputs\w07_action_playbook_queue.csv: exists
C:\INTERNSHIP\ML-INTERNSHIP\work\figures\w07_baseline_score_distribution.svg: exists
C:\INTERNSHIP\ML-INTERNSHIP\work\figures\w07_week4_precision_at_k.svg: exists
C:\INTERNSHIP\ML-INTERNSHIP\work\outputs\w07_action_playbook_metrics.json: exists
Flagged, not resolved: staleness was OPPOSITE; volume was MIXED; and the Week 5 grouped-test base rate differs from the full queue.
